[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M02/M02_Lab2_System_Messages_Templates.ipynb)

# M02 Lab 2 — System Messages, Templates & Model Comparison

**DADS 5250: Generative AI in Practice** | Northeastern University

Difficulty: ⭐ | Time: ~10 min

## Learning Objectives

1. Craft effective system messages and role prompts
2. Build reusable prompt templates with parameterization
3. Compare GPT vs Gemini outputs side by side

In [ ]:
!pip install -q openai tiktoken google-genai
!pip install -q git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils

In [ ]:
from openai import OpenAI
from google import genai
from dads5250 import setup_openai, setup_gemini, show_response, compare_responses, show_expected, quiz

client = setup_openai()
gemini = setup_gemini()

---
## 1. System Messages & Role Prompts

The **system message** sets the model's persona, tone, and constraints before the user interacts.

In [ ]:
personas = {
    "Formal Professor": "You are a formal university professor. Use academic language and cite concepts precisely.",
    "Friendly Tutor": "You are a friendly tutor who explains things simply, using analogies and everyday language.",
    "Concise Expert": "You are a concise AI expert. Answer in 2-3 bullet points maximum. No fluff.",
}

question = "What is prompt engineering?"
results = {}

for name, system_msg in personas.items():
    r = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": question},
        ],
        max_tokens=200,
    )
    results[name] = r.choices[0].message.content.strip()

compare_responses(results)

---
## 2. Prompt Templates

Reusable prompt templates let you parameterize prompts for different inputs.

In [ ]:
def classify_with_template(text: str, categories: list[str]) -> str:
    """Reusable classification prompt template."""
    cats = ", ".join(categories)
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": f"Classify the following text into one of these categories: {cats}. Respond with just the category name."},
            {"role": "user", "content": text},
        ],
        max_tokens=20,
    )
    return response.choices[0].message.content.strip()

samples = [
    "My laptop screen cracked after dropping it",
    "How do I reset my password?",
    "I want a refund for my order #12345",
]
categories = ["Hardware Issue", "Account Help", "Billing/Refund", "General Inquiry"]

for s in samples:
    result = classify_with_template(s, categories)
    print(f"'{s}' -> {result}")

---
## 3. GPT vs Gemini: Side-by-Side

Let's compare how OpenAI's GPT and Google's Gemini handle the same prompt.

In [ ]:
prompt = "Explain the concept of 'attention' in transformer models in 3 sentences."

gpt_r = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=200,
)

gemini_r = gemini.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
)

compare_responses({
    "GPT-4.1-mini": gpt_r.choices[0].message.content,
    "Gemini 2.5 Flash": gemini_r.text,
})

---
## 4. Knowledge Check

In [ ]:
quiz([
    {
        "q": "What is the purpose of a system message?",
        "options": [
            "To authenticate the API key",
            "To set the model's persona, constraints, and behavior",
            "To count tokens in the prompt",
            "To select which model to use"
        ],
        "answer": 1,
        "explanation": "System messages define the assistant's role, tone, and constraints before the conversation begins."
    }
])

---
## 5. Hands-on: GPT vs Gemini Comparison (Observational)

Run 5 different prompts on both models and document your observations.

In [ ]:
test_prompts = [
    "Translate 'Good morning, how are you?' to French, Spanish, and Japanese.",
    "Write a haiku about machine learning.",
    "What are the top 3 causes of climate change? Be concise.",
    "If a train leaves at 3pm going 60mph and another at 4pm going 80mph, when does the second catch up? Think step by step.",
    "Summarize the concept of supply and demand in exactly 2 sentences.",
]

for i, prompt in enumerate(test_prompts, 1):
    gpt_r = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=200,
    )
    gemini_r = gemini.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )
    print(f"--- Prompt {i} ---")
    compare_responses({
        "GPT-4.1-mini": gpt_r.choices[0].message.content,
        "Gemini 2.5 Flash": gemini_r.text,
    })
    print()

**Your Observations:** (double-click to edit)

| Prompt Type | Better Model | Why? |
|------------|-------------|------|
| Translation | | |
| Creative | | |
| Factual | | |
| Reasoning | | |
| Summarization | | |

---
## Summary

- **System messages** control persona, tone, and constraints
- **Prompt templates** make prompts reusable and parameterized
- **Model comparison**: GPT and Gemini have different strengths — always test both

---

## Assignment M02

Design a prompt for a **real-world task** (customer support, data extraction, email drafting, etc.).

**Test 3 techniques:** zero-shot, few-shot, chain-of-thought

**Submit:**
- Your 3 prompt versions + outputs
- Analysis (1 paragraph): Which technique worked best and why?